# Security Notice

This notebook contains code that requires AWS credentials to run. For security and compliance reasons, 
actual credentials have been removed from this submission and replaced with placeholders.

## For Instructors/Graders:
- All code logic and implementation details are preserved
- Any cell that would have contained real credentials is clearly marked
- The code will not run as-is without valid credentials
- If you need to see the fully functional code, please contact me to arrange a secure demonstration or review

## To use this notebook:
1. Replace all placeholder credentials with valid AWS credentials
2. Ensure your AWS account has the necessary permissions for:
   - Elasticsearch Service
   - DynamoDB
   - Lambda
   - Any other AWS services used in the implementation
3. Run all cells in order

This approach follows security best practices and prevents accidental exposure of sensitive information.

In [ ]:
import requests

# Replace with your API key
API_KEY = 'YOUR_API_KEY'  # Replace with your actual yelp api key
ENDPOINT = 'https://api.yelp.com/v3/businesses/search'
HEADERS = {
    'Authorization': f'Bearer {API_KEY}'
}

# Test parameters
params = {
    'term': 'Chinese restaurants',
    'location': 'Manhattan',
    'limit': 5  # Retrieve only 5 records for testing
}

# Send request
response = requests.get(ENDPOINT, headers=HEADERS, params=params)

# Print results
if response.status_code == 200:
    data = response.json()
    print("Data retrieved successfully!")
    print("\nRetrieved restaurants:")
    for business in data['businesses']:
        print(f"- {business['name']}: {business['rating']} stars, {business['review_count']} reviews")
else:
    print(f"Request failed: {response.status_code}")
    print(response.text)


成功获取数据！

获取的餐厅：
- Blue Willow 夜来湘: 4.4星, 1348条评论
- Tipsy Shanghai: 4.3星, 111条评论
- The Best Sichuan: 4.4星, 212条评论
- Dim Sum Palace: 4.1星, 1849条评论
- Chi Restaurant & Bar: 4.5星, 410条评论


In [3]:
import requests
import json
from IPython.display import display, JSON

In [5]:
# Define cuisine types to search
CUISINE_TYPES = [
    'Chinese',
    'Japanese',
    'Italian',
    'American',
    'Mexican'
]

def search_restaurants(cuisine_type, offset=0):
    """Search for specific type of restaurants"""
    params = {
        'term': f'{cuisine_type} restaurants',
        'location': 'Manhattan',
        'limit': 50,  # Yelp API returns max 50 records per request
        'offset': offset
    }
    
    response = requests.get(ENDPOINT, headers=HEADERS, params=params)
    return response.json()

# Test getting restaurants for one cuisine type
test_cuisine = 'Chinese'
results = search_restaurants(test_cuisine)

# Display the number of restaurants retrieved
print(f"Retrieved {len(results['businesses'])} {test_cuisine} restaurants")

# Display details for first 5 restaurants
print("\nFirst 5 restaurants details:")
for restaurant in results['businesses'][:5]:
    print(f"\n🏮 {restaurant['name']}")
    print(f"   ID: {restaurant['id']}")
    print(f"   Rating: {'⭐' * int(restaurant['rating'])} ({restaurant['rating']})")
    print(f"   Review Count: {restaurant['review_count']}")
    print(f"   Address: {' '.join(restaurant['location']['display_address'])}")
    print(f"   Zip Code: {restaurant['location']['zip_code']}")
    print(f"   Coordinates: {restaurant['coordinates']['latitude']}, {restaurant['coordinates']['longitude']}")

Retrieved 50 Chinese restaurants

First 5 restaurants details:

🏮 Blue Willow 夜来湘
   ID: XsXLVWr1UZWVhKThNvNiaA
   Rating: ⭐⭐⭐⭐ (4.4)
   Review Count: 1348
   Address: 40 W 56th St New York, NY 10019
   Zip Code: 10019
   Coordinates: 40.76292, -73.976546

🏮 The Best Sichuan
   ID: WVsPyl-DGRu0O_rQYdIMVQ
   Rating: ⭐⭐⭐⭐ (4.4)
   Review Count: 212
   Address: 47 W 39th St New York, NY 10018
   Zip Code: 10018
   Coordinates: 40.75277108646358, -73.9845295

🏮 Dim Sum Palace
   ID: bVkjBJlAIwKAj9Aw1KVWjA
   Rating: ⭐⭐⭐⭐ (4.1)
   Review Count: 1849
   Address: 334 W 46th St New York, NY 10036
   Zip Code: 10036
   Coordinates: 40.76015, -73.9893699

🏮 Chi Restaurant & Bar
   ID: 83zDAk03lFXkgImgkLhT9w
   Rating: ⭐⭐⭐⭐ (4.5)
   Review Count: 410
   Address: 492 9th Ave New York, NY 10018
   Zip Code: 10018
   Coordinates: 40.75547, -73.99439

🏮 Joe's Shanghai
   ID: 0CjK3esfpFcxIopebzjFxA
   Rating: ⭐⭐⭐ (3.8)
   Review Count: 7522
   Address: 46 Bowery St New York, NY 10013
   Zip Code: 1001

In [7]:
# Check complete data structure for one restaurant
sample_restaurant = results['businesses'][0]
print("Restaurant Data Structure:")
display(JSON(sample_restaurant))

# Verify if we have all required fields
required_fields = [
    'business_id',
    'name',
    'address',
    'coordinates',
    'review_count',
    'rating',
    'zip_code'
]

print("\nField Verification:")
for field in required_fields:
    if field == 'business_id':
        exists = 'id' in sample_restaurant
    elif field == 'address':
        exists = 'location' in sample_restaurant and 'display_address' in sample_restaurant['location']
    elif field == 'zip_code':
        exists = 'location' in sample_restaurant and 'zip_code' in sample_restaurant['location']
    else:
        exists = field in sample_restaurant
    
    print(f"{field}: {'✅' if exists else '❌'}")

Restaurant Data Structure:


<IPython.core.display.JSON object>


Field Verification:
business_id: ✅
name: ✅
address: ✅
coordinates: ✅
review_count: ✅
rating: ✅
zip_code: ✅


In [9]:
import time
from datetime import datetime

def collect_restaurants(cuisine_type, target_count=1000):
    """
    Collect restaurants for a specific cuisine type
    """
    all_restaurants = []
    offset = 0
    collected_ids = set()  # To prevent duplicates
    
    print(f"Collecting {cuisine_type} restaurants...")
    
    while len(all_restaurants) < target_count:
        try:
            # Get results
            results = search_restaurants(cuisine_type, offset)
            
            if 'businesses' not in results or not results['businesses']:
                print(f"No more {cuisine_type} restaurants found. Total collected: {len(all_restaurants)}")
                break
            
            # Process results
            for restaurant in results['businesses']:
                if restaurant['id'] not in collected_ids:
                    collected_ids.add(restaurant['id'])
                    all_restaurants.append(restaurant)
                    print(f"Collected {len(all_restaurants)}: {restaurant['name']}", end='\r')
            
            offset += 50  # Move to next page
            time.sleep(1)  # Respect API rate limits
            
        except Exception as e:
            print(f"\nError during collection: {str(e)}")
            time.sleep(2)  # Wait longer if there's an error
        
        if len(all_restaurants) >= target_count:
            break
    
    print(f"\nFinished collecting {len(all_restaurants)} {cuisine_type} restaurants")
    return all_restaurants

# Test with one cuisine type
test_results = collect_restaurants('Chinese', target_count=100)  # Let's start with 100 for testing

# Display summary of collected data
print("\nCollection Summary:")
print(f"Total restaurants collected: {len(test_results)}")
print("\nSample of collected data:")
for restaurant in test_results[:3]:
    print(f"\n{restaurant['name']}")
    print(f"Rating: {restaurant['rating']}")
    print(f"Reviews: {restaurant['review_count']}")

Collected 100: Lilli & Loo川聚院r West Side Baresrkwn
Finished collecting 100 Chinese restaurants

Collection Summary:
Total restaurants collected: 100

Sample of collected data:

Blue Willow 夜来湘
Rating: 4.4
Reviews: 1348

The Best Sichuan
Rating: 4.4
Reviews: 212

Dim Sum Palace
Rating: 4.1
Reviews: 1849


In [13]:
!pip uninstall -y boto3 botocore aiobotocore

Found existing installation: boto3 1.36.22
Uninstalling boto3-1.36.22:
  Successfully uninstalled boto3-1.36.22
Found existing installation: botocore 1.36.22
Uninstalling botocore-1.36.22:
  Successfully uninstalled botocore-1.36.22
Found existing installation: aiobotocore 2.12.3
Uninstalling aiobotocore-2.12.3:
  Successfully uninstalled aiobotocore-2.12.3


In [15]:
!pip install boto3==1.34.41 botocore==1.34.41 aiobotocore==2.12.3

   ---------------------------------------- 0.0/139.3 kB ? eta -:--:--
   -------------------------------------- - 133.1/139.3 kB 4.0 MB/s eta 0:00:01
   ---------------------------------------- 139.3/139.3 kB 2.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
    --------------------------------------- 0.2/12.0 MB 3.5 MB/s eta 0:00:04
   - -------------------------------------- 0.3/12.0 MB 3.5 MB/s eta 0:00:04
   - -------------------------------------- 0.5/12.0 MB 3.5 MB/s eta 0:00:04
   -- ------------------------------------- 0.7/12.0 MB 3.7 MB/s eta 0:00:04
   -- ------------------------------------- 0.9/12.0 MB 3.8 MB/s eta 0:00:03
   --- ------------------------------------ 1.0/12.0 MB 3.4 MB/s eta 0:00:04
   --- ------------------------------------ 1.2/12.0 MB 3.6 MB/s eta 0:00:03
   ---- ----------------------------------- 1.4/12.0 MB 3.7 MB/s eta 0:00:03
   ---- ----------------------------------- 1.4/12.0 MB 3.5 MB/s eta 0:00:04
   ---

In [19]:
import boto3
import botocore

print(f"boto3 version: {boto3.__version__}")
print(f"botocore version: {botocore.__version__}")

boto3 version: 1.34.41
botocore version: 1.34.41


In [ ]:
import os


os.environ['AWS_ACCESS_KEY_ID'] = "YOUR_ACCESS_KEY_ID"  # Replace with your actual AWS Access Key ID
os.environ['AWS_SECRET_ACCESS_KEY'] = "YOUR_SECRET_ACCESS_KEY"  # Replace with your actual AWS Secret Access Key
os.environ['AWS_DEFAULT_REGION'] = 'us-east-1' 

In [23]:
import boto3

try:
    # try to display table in dynamodb
    dynamodb = boto3.client('dynamodb')
    tables = dynamodb.list_tables()
    print("AWS configuration successful!")
    print("Existing tables:", tables['TableNames'])
except Exception as e:
    print("Error connecting to AWS:", str(e))


AWS configuration successful!
Existing tables: []


In [25]:
import boto3
from botocore.exceptions import ClientError
from datetime import datetime

# Initialize DynamoDB client
dynamodb = boto3.resource('dynamodb')

def create_restaurant_table():
    """
    Create DynamoDB table for restaurants
    """
    try:
        table = dynamodb.create_table(
            TableName='yelp-restaurants',
            KeySchema=[
                {
                    'AttributeName': 'business_id',
                    'KeyType': 'HASH'  # Partition key
                }
            ],
            AttributeDefinitions=[
                {
                    'AttributeName': 'business_id',
                    'AttributeType': 'S'  # String
                }
            ],
            ProvisionedThroughput={
                'ReadCapacityUnits': 5,
                'WriteCapacityUnits': 5
            }
        )
        print("Creating table...")
        table.meta.client.get_waiter('table_exists').wait(TableName='yelp-restaurants')
        print("Table created successfully!")
        return table
    except ClientError as e:
        if e.response['Error']['Code'] == 'ResourceInUseException':
            print("Table already exists")
            return dynamodb.Table('yelp-restaurants')
        else:
            print(f"Error creating table: {str(e)}")
            raise

# Create the table
table = create_restaurant_table()

Creating table...
Table created successfully!


In [27]:
# Verify table creation
dynamodb_client = boto3.client('dynamodb')
tables = dynamodb_client.list_tables()
print("Current tables:", tables['TableNames'])

Current tables: ['yelp-restaurants']


In [ ]:
def clear_table():
    """
    Delete and recreate the yelp-restaurants table
    """
    try:
        # Delete existing table
        table = dynamodb.Table('yelp-restaurants')
        table.delete()
        print("Deleting existing table...")
        
        # Wait for table to be deleted
        table.wait_until_not_exists()
        print("Table deleted successfully")
        
        # Recreate table
        create_restaurant_table()
        print("Table recreated and ready for new data")
        
    except Exception as e:
        print(f"Error during table reset: {str(e)}")

# Clear the table
print("Starting table reset...")
clear_table()

# Verify table is empty
def verify_table_empty():
    table = dynamodb.Table('yelp-restaurants')
    response = table.scan()
    count = len(response['Items'])
    print(f"Current item count in table: {count}")
    return count == 0

# Verify
print("\nVerifying table is empty...")
if verify_table_empty():
    print("Table is empty and ready for new data")
else:
    print("Warning: Table still contains some items")

Starting table reset...
Deleting existing table...


In [45]:
clear_table()

Deleting existing table...
Table deleted successfully
Creating table...
Table created successfully!
Table recreated and ready for new data


In [ ]:
import time
import requests
import boto3
from datetime import datetime, timezone
from decimal import Decimal

# Yelp API Configuration
YELP_API_KEY = API_KEY
ENDPOINT = "https://api.yelp.com/v3/businesses/search"
HEADERS = {"Authorization": f"Bearer {YELP_API_KEY}"}

# DynamoDB Configuration
AWS_REGION = "us-east-1"
DYNAMODB_TABLE_NAME = "yelp-restaurants"

# Connect to DynamoDB
dynamodb = boto3.resource("dynamodb", region_name=AWS_REGION)
table = dynamodb.Table(DYNAMODB_TABLE_NAME)

# Manhattan ZIP codes
MANHATTAN_ZIP_CODES = {
    "10001", "10002", "10003", "10004", "10005", "10006", "10007", "10009", "10010",
    "10011", "10012", "10013", "10014", "10016", "10017", "10018", "10019", "10020",
    "10021", "10022", "10023", "10024", "10025", "10026", "10027", "10028", "10029",
    "10030", "10031", "10032", "10033", "10034", "10035", "10036", "10037", "10038",
    "10039", "10040", "10044", "10065", "10069", "10075", "10128", "10280", "10281", "10282"
}

# Collect only 3 cuisines, 50 restaurants per cuisine
CUISINE_SEARCHES = {
    'Chinese': ['Chinese food', 'Dim Sum', 'Szechuan'],
    'Japanese': ['Japanese food', 'Sushi', 'Ramen'],
    'Italian': ['Italian food', 'Pizza', 'Pasta']
}

# Check if ZIP Code belongs to Manhattan
def is_manhattan_zipcode(zip_code):
    return zip_code in MANHATTAN_ZIP_CODES

# Check if restaurant already exists in DynamoDB
def is_duplicate(business_id):
    try:
        response = table.get_item(Key={"business_id": business_id})
        return "Item" in response
    except Exception as e:
        print(f"AWS Error: {e}")
        return False

# Insert restaurant data into DynamoDB
def insert_to_dynamodb(restaurant):
    try:
        zip_code = restaurant["location"].get("zip_code", "Unknown")

        # Ensure ZIP Code is in Manhattan
        if not is_manhattan_zipcode(zip_code):
            print(f"Skipping {restaurant['name']} - ZIP {zip_code} is not in Manhattan")
            return False

        # Convert coordinates to Decimal to comply with DynamoDB
        coordinates = {
            "latitude": Decimal(str(restaurant["coordinates"]["latitude"])) if "latitude" in restaurant["coordinates"] else None,
            "longitude": Decimal(str(restaurant["coordinates"]["longitude"])) if "longitude" in restaurant["coordinates"] else None
        }

        item = {
            "business_id": restaurant["id"],
            "name": restaurant["name"],
            "address": ", ".join(restaurant["location"]["display_address"]),
            "coordinates": coordinates,
            "review_count": restaurant["review_count"],
            "rating": Decimal(str(restaurant["rating"])),  # Convert float to Decimal
            "zip_code": zip_code,
            "insertedAtTimestamp": datetime.now(timezone.utc).isoformat()  # Use timezone-aware UTC timestamp
        }
        table.put_item(Item=item)
        return True
    except Exception as e:
        print(f"Error inserting {restaurant['id']}: {str(e)}")
        return False

# Collect Yelp restaurant data
def collect_restaurants_by_cuisine(cuisine_type, search_terms, target_count=50):
    collected_restaurants = []
    collected_ids = set()
    sort_methods = ["rating", "review_count", "best_match"]
    
    for search_term in search_terms:
        for sort_by in sort_methods:
            if len(collected_restaurants) >= target_count:
                break

            print(f"\nSearching: {search_term} (Sort: {sort_by})")
            offset = 0

            while offset < 200 and len(collected_restaurants) < target_count:
                params = {
                    "term": search_term,
                    "location": "New York, NY",
                    "limit": 50,
                    "offset": offset,
                    "sort_by": sort_by
                }

                try:
                    response = requests.get(ENDPOINT, headers=HEADERS, params=params)
                    results = response.json()
                    
                    # Print API response for debugging
                    print(f"API Request: {params}")
                    print(f"API Response: {results}")

                    if "error" in results:
                        print(f"API Error: {results['error']}")
                        break

                    if "businesses" not in results or not results["businesses"]:
                        print(f"⚠️ No results for {search_term} with {sort_by}. Trying next search term.")
                        break

                    for restaurant in results["businesses"]:
                        business_id = restaurant["id"]
                        zip_code = restaurant["location"].get("zip_code", "Unknown")

                        # Filter by Manhattan ZIP code and avoid duplicates
                        if (business_id not in collected_ids and 
                            not is_duplicate(business_id) and 
                            is_manhattan_zipcode(zip_code)):

                            collected_ids.add(business_id)
                            collected_restaurants.append(restaurant)
                            print(f"Collected {len(collected_restaurants)}: {restaurant['name']}", end="\r")

                    offset += 50
                    time.sleep(1)

                except Exception as e:
                    print(f"Error: {str(e)} - Retrying...")
                    time.sleep(2)
                    continue
                
                if len(collected_restaurants) >= target_count:
                    break

    return collected_restaurants

# Execute small collection
def run_small_collection():
    total_collected = 0
    for cuisine_type, search_terms in CUISINE_SEARCHES.items():
        print(f"\nStarting collection for: {cuisine_type}")
        restaurants = collect_restaurants_by_cuisine(cuisine_type, search_terms, target_count=50)
        
        # Insert data into DynamoDB
        inserted_count = 0
        for restaurant in restaurants:
            if insert_to_dynamodb(restaurant):
                inserted_count += 1

        print(f"\nFinished collecting {inserted_count} {cuisine_type} restaurants.")
        total_collected += inserted_count
        time.sleep(2)

    print(f"\nTotal unique restaurants collected: {total_collected}")

# Start collection process
print("Starting small Yelp restaurant collection for Manhattan...")
run_small_collection()
print("Collection complete!")
